# Task #5 — Run v1 extraction pipeline on a sample

Runs the team's planned GLiNER v1 extraction step on a reproducible random sample of 100 cleaned 2025 NEISS narratives.

Output: one extracted object per narrative in `task5_sample_predictions.csv`.

Product_1 is retained for later comparison. This task does not manually label or score predictions.


In [ ]:
!pip -q install "gliner>=0.2" pandas


## 1. Upload the prepared sample

Upload `task5_sample_input.csv` when prompted.


In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
sample = pd.read_csv("task5_sample_input.csv")
print("Rows:", len(sample))
sample.head()


## 2. Load GLiNER

The entity labels come from the team's model logic plan.


In [ ]:
from gliner import GLiNER

MODEL_NAME = "urchade/gliner_medium-v2.1"
LABELS = ["object", "tool", "furniture", "sports equipment", "structure"]
THRESHOLD = 0.30

model = GLiNER.from_pretrained(MODEL_NAME)
print("Loaded:", MODEL_NAME)


## 3. Extract one object per narrative

When several entities are returned, keep the highest-confidence candidate. If none clears the threshold, output `OTHER`.


In [ ]:
def extract_one_object(text):
    if pd.isna(text) or not str(text).strip():
        return "OTHER", None, None

    entities = model.predict_entities(str(text), LABELS, threshold=THRESHOLD)
    if not entities:
        return "OTHER", None, None

    best = max(entities, key=lambda e: float(e["score"]))
    return best["text"].strip(), best["label"], float(best["score"])

pred = sample["event_text"].apply(extract_one_object)
sample["extracted_object"] = pred.apply(lambda x: x[0])
sample["extracted_entity_type"] = pred.apply(lambda x: x[1])
sample["extraction_confidence"] = pred.apply(lambda x: x[2])

sample[["case_number","event_text","extracted_object",
        "extracted_entity_type","extraction_confidence",
        "product_code","product_label"]].head(20)


## 4. Save Task #5 output


In [ ]:
OUTPUT = "task5_sample_predictions.csv"
cols = [
    "case_number","narrative","narrative_normalized","event_text",
    "extracted_object","extracted_entity_type","extraction_confidence",
    "product_code","product_label","product_2_code","product_3_code"
]
sample[cols].to_csv(OUTPUT, index=False)

print("Saved:", OUTPUT)
print("Rows:", len(sample))
print("OTHER predictions:", (sample["extracted_object"] == "OTHER").sum())
files.download(OUTPUT)
